# AI Lab - Practical Question No. 4
## Goal-Based Navigation Agent: 5x5 Grid Matrix Simulation

- **Syllabus Area**: Intelligent Agents
- **Difficulty**: 🟡 Intermediate
- **Practical Type**: Algorithm Design
- **Core AI Concept**: Goal-Based Agent, Distance Metrics (Manhattan & Euclidean), Greedy Coordinate Reduction
- **Python Concepts Tested**: Tuples, coordinate geometry, modular functions, object-oriented design
- **Complexity**: Time Complexity: $O(D)$ | Space Complexity: $O(D)$ where $D = |x_1 - x_2| + |y_1 - y_2|$ is Manhattan distance

---

### Problem Statement
Develop a Python-based **Goal-Based Agent** tasked with finding a specific charging station in a 5x5 grid matrix.
The agent knows the coordinates of the goal and uses a simple distance-reducing heuristic to select orthogonal actions (`Up`, `Down`, `Left`, `Right`) that minimize the Manhattan distance to the goal.

**Task Requirements**:
1. Define a 5x5 matrix with a randomly placed goal (charging station).
2. Implement a function to calculate Manhattan distance.
3. Select actions that strictly reduce this distance.
4. Track and print the agent's path.

**Sample Input**: Start: `(0, 0)`, Goal: `(3, 4)`  
**Sample Output**: `Path: [(0,0), (1,0), (2,0), (3,0), (3,1), (3,2), (3,3), (3,4)]`

**Optional Challenge**: Prevent diagonal movement explicitly in the successor function.

### Agent PEAS Specification

| PEAS Component | Specification for Question 4 |
| :--- | :--- |
| **P**erformance Measure | Reach the charging station in minimum orthogonal steps; strictly reduce distance at every move; zero boundary collisions |
| **E**nvironment | 5x5 discrete matrix grid: `[0, 4] x [0, 4]` without obstacles |
| **A**ctuators | Orthogonal navigation: `Up` $(-1, 0)$, `Down` $(+1, 0)$, `Left` $(0, -1)$, `Right` $(0, +1)$ |
| **S**ensors | Coordinate sensors providing agent current location $(x, y)$ and target goal coordinates $(g_x, g_y)$ |

### Mathematical Distance Metrics

1. **Manhattan Distance ($L_1$ Norm / Taxicab Metric)**:
   $$d_M((x_1, y_1), (x_2, y_2)) = |x_1 - x_2| + |y_1 - y_2|$$
   Measures the exact shortest path length on an obstacle-free 4-connected grid.

2. **Euclidean Distance ($L_2$ Norm / Straight-Line Metric)**:
   $$d_E((x_1, y_1), (x_2, y_2)) = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$
   By the Minkowski inequality, $d_E \le d_M$ always holds for any coordinate pair on the grid.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on sys.path when running from notebooks/
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.goal_based_agent import (
    GoalBasedNavigationAgent,
    GridEnvironment,
    run_goal_based_simulation,
    manhattan_distance,
    euclidean_distance,
    get_successors,
    is_diagonal_movement,
    validate_orthogonal_transition,
    generate_random_coordinate,
    ACTION_UP,
    ACTION_DOWN,
    ACTION_LEFT,
    ACTION_RIGHT,
)
print("Goal-Based Navigation Agent modules imported successfully.")

## 1. Simulation: Sample Scenario

- **Start Coordinate**: `(0, 0)`
- **Goal Coordinate**: `(3, 4)`
- **Algorithm**: Greedy Manhattan distance reduction prioritizing orthogonal actions.

In [ ]:
result = run_goal_based_simulation(start_pos=(0, 0), goal_pos=(3, 4))

print("=" * 75)
print("STEP-BY-STEP NAVIGATION TRACE:")
print("=" * 75)
for step in result.history:
    print(step.format_trace())

print("\n" + "-" * 75)
print(f"Sample Output: Path: {result.path}")
print("-" * 75)
print(result.summary())

print("\n5x5 Grid Matrix Visualization:")
print(result.render_grid())

## 2. Dynamic Scenario: Random Goal Placement

Demonstrating requirement 1: Defining a 5x5 matrix with a randomly placed charging station goal.

In [ ]:
# Reproducible random goal placement using seed=42
random_result = run_goal_based_simulation(
    start_pos=(0, 0),
    random_goal=True,
    seed=42,
)

print(f"Randomly Placed Goal : {random_result.goal_pos}")
print(f"Initial Distance     : {random_result.initial_distance}")
print(f"Path Taken           : {random_result.path}")
print(f"Total Moves          : {random_result.total_steps}")
print("\nGrid Matrix:")
print(random_result.render_grid())

## 3. Optional Challenge: Explicit Diagonal Movement Prevention

In a 4-connected grid, movements are strictly constrained to orthogonal axes ($|\Delta x| + |\Delta y| = 1$).
The successor function explicitly intercepts and rejects any transition with $|\Delta x| > 0$ and $|\Delta y| > 0$.

In [ ]:
current_pos = (2, 2)
print(f"Testing transitions from center cell {current_pos}:\n")

# 1. Valid orthogonal successors generated
successors = get_successors(current_pos, grid_dimensions=(5, 5), allow_diagonal=False)
print("Valid 4-connected successors:")
for act, nxt in successors:
    print(f"  Action: {act:<5} -> Next: {nxt} | d_M: {manhattan_distance(current_pos, nxt)}")

print("\nTesting explicit rejection of diagonal transitions:")
diagonal_tests = [
    ((2, 2), (1, 1), "Up-Left"),
    ((2, 2), (1, 3), "Up-Right"),
    ((2, 2), (3, 1), "Down-Left"),
    ((2, 2), (3, 3), "Down-Right"),
]
for src, dst, label in diagonal_tests:
    try:
        validate_orthogonal_transition(src, dst)
    except ValueError as err:
        print(f"  [BLOCKED] {label} {src} -> {dst}: {err}")

## 4. Distance Metric Comparison: Manhattan vs. Euclidean

Demonstrating the mathematical relationship $d_E \le d_M$ across every coordinate in the agent's path.

In [ ]:
goal = (3, 4)
print(f"{'Step':<5} | {'Coordinate':<12} | {'Manhattan (L1)':<15} | {'Euclidean (L2)':<15} | {'d_E <= d_M'}")
print("-" * 65)
for i, coord in enumerate(result.path):
    d_m = manhattan_distance(coord, goal)
    d_e = euclidean_distance(coord, goal)
    print(f"{i:<5} | {str(coord):<12} | {d_m:<15} | {d_e:<15.4f} | {d_e <= d_m}")

## 5. Viva Voce Reference & Study Guide

### 1. Difference Between Goal-Based and Reflex Agents
- **Simple Reflex Agent**:
  - Operates purely on condition-action rules: `IF condition THEN action`.
  - Completely lacks an explicit goal representation.
  - If the goal moves (e.g., charging station relocated from `(3, 4)` to `(1, 1)`), the reflex agent has no mechanism to adapt without completely rewriting its hardcoded rule table.
- **Goal-Based Agent**:
  - Combines current state knowledge with an explicit *goal description*.
  - Evaluates actions by asking: *"Will this action lead toward achieving the goal?"*
  - Highly flexible and modular: changing the goal coordinate immediately adapts the agent's behavior without modifying the decision-making logic.

### 2. Search vs. Heuristics
- **Search**:
  - The process of exploring the state space from an initial state via successor operators to find a sequence of actions that achieves a goal.
  - Uninformed search (BFS, DFS) explores blindly without cost estimation.
- **Heuristics ($h(n)$)**:
  - An evaluation function providing an informed estimate of the cost or distance from state $n$ to the goal.
  - Here, the Manhattan distance serves as the heuristic function guiding greedy coordinate reduction.

### 3. Manhattan vs. Euclidean Distance
- **Manhattan Distance ($L_1$ Norm)**: $d_M = |x_1 - x_2| + |y_1 - y_2|$. Accurate and exact for 4-connected orthogonal grids (city streets, grid chips, warehouse robots).
- **Euclidean Distance ($L_2$ Norm)**: $d_E = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$. Accurate for continuous 2D Euclidean planes.
- **Admissibility**: Because $d_E \le d_M$, Euclidean distance never overestimates the true grid distance, making it an admissible heuristic for A* search, while Manhattan distance is both admissible and perfectly informed (exact cost) on obstacle-free grids.

### 4. Complexity Analysis
- **Time Complexity**: $O(D)$ where $D = |x_1 - x_2| + |y_1 - y_2|$ is the Manhattan distance. Each move strictly reduces distance by 1.
- **Space Complexity**: $O(D)$ to store the path coordinate tuples from start to goal.